In [ ]:
import random
import copy
import torch
import pickle
import os
import matplotlib.pyplot as plt
import numpy as np

from collections import defaultdict

from causal_gym import AntMazePCH
from causal_rl.algo.imitation.imitate import *
from causal_rl.algo.imitation.finetune import *
from causal_rl.algo.imitation.gail.core_net import *
from causal_rl.algo.imitation.gail.causal_gail import *

In [ ]:
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

In [ ]:
num_steps = 1000
hidden_dims = {'O'}

In [ ]:
# for eval: corrupted W, O hidden
eval_env = AntMazePCH(env_id='antmaze-large-navigate-singletask-task1-v0', num_steps=num_steps, expert_mode=False)

In [ ]:
# load model
MODEL_PATH = '/home/et2842/causal/causalrl/models/cbc_k100_antlarge.pt'
ckpt = torch.load(MODEL_PATH, map_location=device, weights_only=False)

action_bounds = (ckpt['action_bounds_low'], ckpt['action_bounds_high'])

causal_bc_model = ContinuousPolicyNN(
    input_dim=ckpt['input_dim'],
    action_dim=ckpt['num_actions'],
    hidden_dim=ckpt['hidden_dim'],
    num_blocks=ckpt['num_blocks'],
    dropout=ckpt['dropout'],
    layernorm=ckpt['layernorm'],
    final_tanh=ckpt['final_tanh'],
    action_bounds=action_bounds,
).to(device)

causal_bc_model.load_state_dict(ckpt['state_dict'])
causal_bc_model.eval()

slots = ckpt['slots']
Z_trim = ckpt['Z_trim']
dims = ckpt['dims']
lookback = ckpt['lookback']

causal_bc_policy = shared_policy_fn_long_horizon(causal_bc_model, slots, Z_trim, continuous=True, device=device)
causal_bc_policies = make_shared_policy_dict(causal_bc_policy)

In [ ]:
num_eval_eps = 1000

causal_bc_returns = collect_imitator_trajectories(
    env=eval_env,
    policies=causal_bc_policies,
    num_episodes=num_eval_eps,
    max_steps=num_steps,
    hidden_dims=hidden_dims,
    show_progress=True
)

len(causal_bc_returns)

In [ ]:
causal_bc_episode_rewards = defaultdict(float)
for rec in causal_bc_returns:
    ep = rec['episode']
    causal_bc_episode_rewards[ep] += float(rec['reward'])

causal_bc_rewards = [causal_bc_episode_rewards[e] for e in range(num_eval_eps)]
sum(causal_bc_rewards) / num_eval_eps

In [ ]:
mean_reward = np.mean(causal_bc_rewards)
std_reward = np.std(causal_bc_rewards)

print(f"E[Y]          = {mean_reward:.4f}")
print(f"Std[Y]        = {std_reward:.4f}")
print(f"E[Y] ± Std[Y] = {mean_reward:.4f} ± {std_reward:.4f}")

In [ ]:
# success rate: % of episodes solved in under 1000 steps
ep_lengths = defaultdict(int)
for rec in causal_bc_returns:
    ep_lengths[rec['episode']] += 1

lengths = np.array([ep_lengths[e] for e in range(num_eval_eps)])
successes = lengths < num_steps
success_rate = successes.mean()
se = np.sqrt(success_rate * (1 - success_rate) / num_eval_eps)

print(f"Success rate   = {100 * success_rate:.2f}% ({successes.sum()}/{num_eval_eps} episodes)")
print(f"Std error      = {100 * se:.2f}%")

In [ ]:
# successful episode lengths
success_lengths = lengths[successes]

if len(success_lengths) > 0:
    print(f"Successful episode lengths (n={len(success_lengths)}):")
    print(f"  Mean   = {np.mean(success_lengths):.2f}")
    print(f"  Std    = {np.std(success_lengths):.2f}")
    print(f"  Median = {np.median(success_lengths):.0f}")
    print(f"  Min    = {np.min(success_lengths)}")
    print(f"  Max    = {np.max(success_lengths)}")
    print(f"  25th%  = {np.percentile(success_lengths, 25):.0f}")
    print(f"  75th%  = {np.percentile(success_lengths, 75):.0f}")
else:
    print("No episodes were solved.")